# 04 · Checking the parser against someone else's

Notebook 01 compared the numpy decoder with a `struct` parse that I also wrote. Both come from my reading of the specification, so a shared misunderstanding would pass both. This notebook compares everything against an **independent implementation**: [itch-book](https://github.com/groovg/itch-book) (MIT licence, a C++ core with a Python interface), run on exactly the same bytes.

It compares order messages field by field, the remaining shares after every execute and cancel, trades by match number, and the best bid and offer for the 25 busiest symbols. The check is a module (`itch_forensics.crosscheck`), so the notebook runs it rather than pasting results.

In [1]:
import datetime as dt, pathlib, tempfile
import numpy as np
import pandas as pd
from itch_forensics import crosscheck
from itch_forensics.book import build_events
from itch_forensics.decode import decode_all
from itch_forensics.framing import index_messages

pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 80); pd.set_option("display.max_rows", 80)
raw = pathlib.Path("../data/itch.bin")
work = pathlib.Path(tempfile.mkdtemp()); whole = work / "whole_messages.bin"
trailing = crosscheck.trimmed_copy(raw, whole)       # itch-book rejects a file that ends mid-message
buf = whole.read_bytes(); idx = index_messages(buf)
print(f"{trailing} trailing bytes trimmed; {len(idx):,} whole messages")

34 trailing bytes trimmed; 8,733,895 whole messages


In [2]:
M, T, B, stats = crosscheck.read_itch_book(whole, dt.date(2019, 12, 30))
print("itch-book decoded", f"{stats['messages']:,}", "messages;", "this package indexed", f"{len(idx):,}", "->", "same" if stats["messages"] == len(idx) else "DIFFERENT")

itch-book decoded 8,733,895 messages; this package indexed 8,733,895 -> same


In [3]:
events = build_events(decode_all(buf, idx))
monitored = events.groupby("locate").size().nlargest(25).index.tolist()
result = crosscheck.compare(buf, M, T, B, monitored)
result

,check,agree,of,all agree
0,A add: stock locate,3179119,3179119,True
1,A add: order reference,3179119,3179119,True
2,A add: side,3179119,3179119,True
3,A add: shares,3179119,3179119,True
4,A add: price,3179119,3179119,True
5,F add: stock locate,752977,752977,True
6,F add: order reference,752977,752977,True
7,F add: side,752977,752977,True
8,F add: shares,752977,752977,True
9,F add: price,752977,752977,True


In [4]:
print(f"{int(result['of'].sum()):,} individual comparisons in {len(result)} checks; "
      f"{int(result.agree.sum()):,} agree; every check fully agrees: {bool(result['all agree'].all())}")

26,308,815 individual comparisons in 35 checks; 26,308,815 agree; every check fully agrees: True


The one place the two differ by design is `C` (execute at a different price): itch-book emits a trade row only for printable ones, 481 of this package's 1,344 `C` messages. The check above accounts for that and confirms the 481 are exactly the printable ones.

## The comparison must be able to fail

Change one price byte in one add message in the bytes this package reads (leaving itch-book's input untouched). The comparison has to notice:

In [5]:
bad = bytearray(buf)
a_pos = int(np.flatnonzero(idx.types == ord("A"))[1000]); off = int(idx.offsets[a_pos])
bad[off + 35] ^= 0x01                                   # last byte of the 4-byte price of the 1,001st add
broken = crosscheck.compare(bytes(bad), M, T, B, monitored)
broken[~broken["all agree"]]

,check,agree,of,all agree
4,A add: price,3179118,3179119,False


Exactly one add price disagrees, and nothing else (one price change in one order does not alter any size or reference). So agreement on the real data is not the comparison being blind.

## What this does and does not show

* It shows that on these 8.7 million messages two independently written parsers and order-book builders produce identical orders, trades, remaining sizes and best bid/offer. That rules out an implementation bug in this package on this data.
* It does not prove the specification was read correctly where both implementations could share a misreading, though a third-party implementation makes that unlikely.
* It says nothing about the rest of the day: the sample ends at 09:30:36.
